## Data Pre-processing Functions<br>
(thanks joel)

In [ ]:
from sklearn.metrics import classification_report
from termcolor import colored
import seaborn as sns
from sklearn.metrics import confusion_matrix
from keras.models import load_model
from matplotlib import pyplot as plt
from keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd
# import csv
import os
# import sys
import cv2
import numpy as np
import tensorflow
from scipy.ndimage.filters import gaussian_filter
from keras.callbacks import ModelCheckpoint
from skimage.transform import rotate
import keract
# pip install keract
#from PIL import Image, ImageFilter

In [ ]:
plt.rcParams['axes.grid'] = False
plt.rcParams["figure.figsize"] = (24, 12)

#Cropping Constants

BINARY_THRESHOLD = 1
RUN_THRESHOLD = 0.14
MIN_HEIGHT = 300

#Artifacts Constants

ARTIFACT_INTENSITY = 220

def load_image(image_path):
    img = cv2.imread(image_path, 1)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img_rgb

def crop_image(image_array, window_size):
    ultrasound_image = find_ultrasound_image(image_array)
    cropped_image = standardize_size(ultrasound_image, window_size)
    return cropped_image

def remove_artifacts(cropped_image):
    artifact_flags = cropped_image >= ARTIFACT_INTENSITY
    artifact_idx = np.where(artifact_flags)
    artifact_idx_list = []
    for i, y in enumerate(artifact_idx[0]):
        x = artifact_idx[1][i]
        artifact_idx_list.append((y, x, cropped_image[y, x]))
    artifact_idx_list.sort(key=lambda tup: tup[2], reverse=False)
    for y, x, _ in artifact_idx_list:
        local_region = cropped_image[(y-1):(y+2), (x-1):(x+2)]
        background_mean = np.mean(sorted(local_region.flatten())[:6])
        cropped_image[y, x] = background_mean
    return cropped_image

def find_ultrasound_image(image_array):
    binary_mask = np.zeros(image_array.shape)
    binary_mask[image_array > BINARY_THRESHOLD] = 1.
    row_mean = np.mean(binary_mask, axis=0)
    col_mean = np.mean(binary_mask, axis=1)
    row_runs = get_bidirectional_runs(row_mean)
    col_runs = get_bidirectional_runs(col_mean)
    row_indices = np.where(row_runs >= np.max(row_runs)*0.75)[0]
    left_bound = row_indices[0]
    right_bound = row_indices[-1]
    col_indices = np.where(col_runs >= np.max(col_runs)*0.75)[0]
    up_bound = col_indices[0]
    down_bound = max(col_indices[-1], up_bound + MIN_HEIGHT)
    cropped_image = image_array[up_bound:down_bound, left_bound:right_bound]
    return cropped_image

def standardize_size(ultrasound_image, window_size):
    resized_image = np.zeros(window_size)
    horizontal_boundary = int((window_size[1] - ultrasound_image.shape[1])/2)
    vertical_boundary = int((window_size[0] - ultrasound_image.shape[0])/2)
    resized_image[vertical_boundary:(vertical_boundary+ultrasound_image.shape[0]),
                  horizontal_boundary:(horizontal_boundary+ultrasound_image.shape[1])] = ultrasound_image
    return resized_image

def get_bidirectional_runs(mean_vector):
    forward_runs = get_runs(mean_vector)
    backward_runs = get_runs(mean_vector[::-1])[::-1]
    bidirectional_runs = forward_runs + backward_runs
    return bidirectional_runs

def get_runs(numeric_vector):
    runs = np.empty(len(numeric_vector), dtype=np.int16)
    curr = 0
    for i, x in enumerate(numeric_vector):
        if x > RUN_THRESHOLD:
            curr += 1
        else:
            curr = 0
        runs[i] = curr
    return runs

def represents_int(s):
    try:
        int(s)
        return True
    except ValueError:
        return False

# Data Preparation

In [ ]:
#df = pd.read_csv("/home/huangs/thyroid_modeling/data/aggregate.csv")
df=pd.read_csv(os.path.join('..','data', 'aggregate.csv'))

sample_id = df.sample_id
image = df.image
rows = len(image)

In [ ]:
window_size = (512, 768)
x = np.zeros((rows, *window_size))

indices=np.array(['l'*20]*rows)

for i in range(rows):

    # x data (the images)
    newstr = ""
    for j in str(image[i]):
        if(j == "n"):
            break
        else:
            newstr += j
    #image_path = '/'+os.path.join('home', 'huangs', 'thyroid_modeling','data', str(sample_id[i]), newstr+'.png')
    image_path=os.path.join('..', 'data', str(sample_id[i]), newstr+'.png')
    image_array = load_image(image_path)
    cropped_image = crop_image(image_array, window_size)
    cleaned_image = remove_artifacts(cropped_image)
    x[i] = cleaned_image
    indices[i]=str(sample_id[i])+'/'+image[i]
#     plt.imshow(cleaned_image,'gray')
#     plt.show()

x = x/255.0

echo_score = df.echo_score
y = pd.DataFrame(echo_score).to_numpy()

indices = np.delete(indices, 0, 0)
x = np.delete(x, 0, 0)
y = np.delete(y, 0, 0)  

y-=1

In [ ]:
# apply gaussian filter (blurring)

# for i in range(rows):
#     x[i] = gaussian_filter(x[i], sigma=2)

Display sample image

In [ ]:
plt.imshow(x[0], 'gray')
plt.show()

plt.imshow(x[1], 'gray')
plt.show()

In [ ]:
train_ratio = 0.70
validation_ratio = 0.15
test_ratio = 0.15

x_train, x_test, y_train, y_test,indices_train,indices_test = train_test_split(x, y,indices, test_size=1-train_ratio, random_state=326, stratify=y)

x_val, x_test, y_val, y_test,indices_val,indices_test = train_test_split(x_test, y_test,indices_test, test_size=test_ratio/(test_ratio + validation_ratio), random_state=326, stratify=y_test)

Skewed Data, hence use class weights

In [ ]:
NumOfZeros = (y_train == 0).sum()
NumOfOnes = (y_train == 1).sum()

class_weight = {
    0: 100/NumOfZeros,
    1: 100/NumOfOnes
}

# 0 1 2 etc required for class weights.

Data Augmentation (Scale width by 60%)  <br>
Data Augmentation (Flip Up Down) <br>
Data Augmentation (Invert left-right) <br>
Data Augmentation (Rotate by 90 degrees anti-clockwise) <br>
Data Augmentation (Rotate by 90 degrees clockwise)

In [ ]:
#Data Augmention to x_train only!

DataAugment = len(x_train)
x_train = np.concatenate((x_train, x_train, x_train,x_train,x_train,x_train))

for i in range(DataAugment):
    AugmentedImg = cv2.resize(x_train[i], dsize=(460, 512))
    AugmentedImg = np.concatenate((np.concatenate(
        (np.zeros((512, 154)), AugmentedImg), axis=1), np.zeros((512, 154))), axis=1)
    x_train[i+DataAugment] = AugmentedImg
    
for i in range(DataAugment):
    AugmentedImg = np.flipud(x_train[i])
    x_train[i+(DataAugment*2)] = AugmentedImg
    
for i in range(DataAugment):
    AugmentedImg = np.fliplr(x_train[i])
    x_train[i+(DataAugment*3)] = AugmentedImg

for i in range(DataAugment):
    AugmentedImg = rotate(x_train[i],angle=90)
    x_train[i+(DataAugment*4)] = AugmentedImg

for i in range(DataAugment):
    AugmentedImg = rotate(x_train[i],angle=270)
    x_train[i+(DataAugment*5)] = AugmentedImg

Display sample augmented images

In [ ]:
print("Original Image:")
plt.imshow(x_train[0], 'gray')
plt.show()

print("Augmented Image (Scale width by 60%)")
plt.imshow(x_train[DataAugment], 'gray')
plt.show()

print("Augmented Image (Flip Up Down)")
plt.imshow(x_train[DataAugment*2], 'gray')
plt.show()

print("Augmented Image (Invert left-right)")
plt.imshow(x_train[DataAugment*3], 'gray')
plt.show()

print("Augmented Image (Rotate by 90 degrees anti-clockwise)")
plt.imshow(x_train[DataAugment*4], 'gray')
plt.show()

print("Augmented Image (Rotate by 90 degrees clockwise)")
plt.imshow(x_train[DataAugment*5], 'gray')
plt.show()

# ALL AUGMENTED DATA ARE IN TRAINING SET ONLY.

In [ ]:
#adjusting/doubling y_train due to data augmentation
y_train = np.concatenate((y_train, y_train,y_train,y_train,y_train,y_train))
indices_train=np.concatenate((indices_train, indices_train,indices_train,indices_train,indices_train,indices_train))



print(x_train.shape)
print(y_train.shape)
print(indices_train.shape)
print(x_val.shape)
print(y_val.shape)
print(indices_val.shape)
print(x_test.shape)
print(y_test.shape)
print(indices_test.shape)

In [ ]:
print(type(x_train))

In [ ]:
#np.save('../x_train.npy', x_train)
#np.save('../y_train.npy', y_train)

# Train & Save model

In [ ]:
import autokeras as ak



In [ ]:
#model_path = '/'+os.path.join('home', 'huangs', 'thyroid_modeling','data', 'Echogenicity_model.h5')
model_path="../data/Echogenicity_model.h5"

In [ ]:
# Initialize the image classifier.
clf = ak.ImageClassifier(overwrite=True, max_trials=2)
# Feed the image classifier with training data.

clf.fit(x_train, y_train, validation_data=(x_val, y_val),epochs=50)
model = clf.export_model()

print(type(model))  # <class 'tensorflow.python.keras.engine.training.Model'>

try:
    model.save("model_autokeras", save_format="tf")
except Exception:
    model.save("model_autokeras.h5")


history = load_model("model_autokeras", custom_objects=ak.CUSTOM_OBJECTS)

predicted_y = history.predict(tf.expand_dims(x_test, -1))
print(predicted_y)




# Evaluate the best model with testing data.
print(history.evaluate(x_test, y_test))





In [ ]:
# #pip intall keract
# #^if no module named keract

# print(indices_train[0])
# print()

# sample_input = np.expand_dims(x_train[0], axis=0)
# activations = keract.get_activations(model, sample_input)
# keract.display_activations(activations, save=False, fig_size=(28, 16.4))

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

# Load model

In [ ]:
model = load_model(model_path, custom_objects={'CustomLayer': CustomLayer})
print("Loaded model from disk")

# Evaluate model

In [ ]:
model.evaluate(x_test,  y_test, verbose=2)

# Results Analysis

In [ ]:
def get_class_prediction(model, x):
    pred = model.predict(x)
    if pred.shape[1] == 1:
        return (pred>0.5).astype(np.int8).squeeze(axis=-1)
    else:
        return pred.argmax(axis=1)

In [ ]:
predicted=get_class_prediction(model, x_test)

NumOfImg=len(y_test)

print("Predicted Results:", end=" ")
for i in predicted:
    print(i, end=" ")
print()
print("Actual Results:  ", end="  ")
actual = y_test
for i in actual:
    for j in i:
        print(j, end=" ")
NumOfImg = len(actual)
ctr = 0
for i in range(NumOfImg):
    if(actual[i] != predicted[i]):
        ctr += 1
print("\nNumber of Correct Predictions:", NumOfImg-ctr, "of", NumOfImg)

In [ ]:
sns.set(font_scale=0.8)
print(colored("Incorrect Classification Report:", attrs=['bold']))
x=np.squeeze(x)
x_test=np.squeeze(x_test)

for i in range(NumOfImg):
    if(actual[i] != predicted[i]):
        plt.figure(figsize=(18, 5))
        plt.grid(False)
        plt.imshow(x_test[i], 'gray')
        plt.show()
        
        image_path="../data/"+indices_test[i]+'.png'
        plt.figure(figsize=(18, 5))
        plt.grid(False)
        image_array = load_image(image_path)
        cropped_image = crop_image(image_array, window_size)
        cleaned_image = remove_artifacts(cropped_image)
        x[i] = cleaned_image
        plt.imshow(x[i], 'gray')
        plt.show()
        
        incorrectResults = "         Predicted label: "+str(predicted[i])
        CorrectResults = "            Actual label: "+str(actual[i][0])
        print(colored(incorrectResults, 'red', attrs=['bold']))
        print(colored(CorrectResults, 'green', attrs=['bold']))
        print("              Image Path:",image_path)

In [ ]:
print(colored("Correct Classification Report:", attrs=['bold']))

for i in range(NumOfImg):
    if(actual[i] == predicted[i]):
        plt.figure(figsize=(18, 5))
        plt.grid(False)
        plt.imshow(x_test[i], 'gray')
        plt.show()
        
        image_path="../data/"+indices_test[i]+'.png'
        plt.figure(figsize=(18, 5))
        plt.grid(False)
        image_array = load_image(image_path)
        cropped_image = crop_image(image_array, window_size)
        cleaned_image = remove_artifacts(cropped_image)
        x[i] = cleaned_image
        plt.imshow(x[i], 'gray')
        plt.show()
        
        correctResults = "         Predicted label: "+str(predicted[i])
        CorrectResults = "            Actual label: "+str(actual[i][0])
        print(colored(correctResults, 'green', attrs=['bold']))
        print(colored(CorrectResults, 'green', attrs=['bold']))
        print("              Image Path:",image_path)

In [ ]:
results = confusion_matrix(actual, predicted)
# print(results)

#Ticket labels - List must be in alphabetical order
sns.set(rc={'figure.figsize': (3, 3)})
sns.set(font_scale=2.4)
ax = sns.heatmap(results, annot=True, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=15)
ax.set_xlabel('\nPredicted Values', fontsize=15)
ax.set_ylabel('Actual Values ', fontsize=15)
ax.xaxis.set_ticklabels(['0', '1'], fontsize=15)
ax.yaxis.set_ticklabels(['0', '1'], fontsize=15)
#use matplotlib.colorbar.Colorbar object
cbar = ax.collections[0].colorbar
# here set the labelsize by 20
cbar.ax.tick_params(labelsize=12)

#Display the visualization of the Confusion Matrix.
plt.show()

In [ ]:
print(classification_report(actual, predicted))